In [1]:
import torch


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [ ]:
import os
import requests
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from model_architecture import DenseUNet
from data_handler import NanoSEMDataset
from metrics import calculate_iou, calculate_dice

# 1. Configuration
BATCH_SIZE = 2
EPOCHS = 100
LEARNING_RATE = 1e-4
IMG_SIZE = (256, 256)
NOTIFY_EVERY = 1  # send an epoch update every N epochs
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# Discord: set DISCORD_WEBHOOK_URL as an environment variable, or paste it below
DISCORD_WEBHOOK_URL = os.environ.get("DISCORD_WEBHOOK_URL", "https://discord.com/api/webhooks/1557954763352440994/GIo7NBA3xOKqmHiKfZaBTA7OqQlwsgM8ajqZLTA3VyCF1yT_esMNu7U7gJOouzoOchmV")


def notify_discord(message):
    """Send a message to Discord. Never lets a network error crash training."""
    if not DISCORD_WEBHOOK_URL or DISCORD_WEBHOOK_URL == "PASTE_URL_HERE":
        return
    try:
        requests.post(DISCORD_WEBHOOK_URL, json={"content": message}, timeout=5)
    except requests.RequestException as e:
        print(f"Discord notification failed: {e}")


# 2. Data
root = r"C:\Users\ullas\Documents\Datasets\MU-KAN-master\inputs\lizi"
train_dataset = NanoSEMDataset(
    image_dir=os.path.join(root, "images", "train"),
    mask_dir=os.path.join(root, "masks", "0", "train"),
    size=IMG_SIZE, augment=True,
)
val_dataset = NanoSEMDataset(
    image_dir=os.path.join(root, "images", "val"),
    mask_dir=os.path.join(root, "masks", "0", "val"),
    size=IMG_SIZE, augment=False,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, pin_memory=False)

# 3. Model, loss, optimizer
model = DenseUNet(in_channels=1, out_channels=1).to(DEVICE)
criterion = nn.BCELoss()  # model ends with Sigmoid
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

best_dice = 0.0

# 4. Training
notify_discord(
    f"🚀 Training started on {DEVICE} | {len(train_dataset)} train / "
    f"{len(val_dataset)} val images | {EPOCHS} epochs"
)

try:
    for epoch in range(EPOCHS):
        # --- Training ---
        model.train()
        epoch_loss = 0.0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)

        for images, masks in progress_bar:
            images, masks = images.to(DEVICE), masks.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            progress_bar.set_postfix(loss=f"{loss.item():.4f}")

        # --- Validation ---
        model.eval()
        val_loss, val_iou, val_dice = 0.0, 0.0, 0.0
        with torch.no_grad():
            for images, masks in val_loader:
                images, masks = images.to(DEVICE), masks.to(DEVICE)
                outputs = model(images)

                val_loss += criterion(outputs, masks).item()
                preds = (outputs > 0.5).float()
                val_iou += float(calculate_iou(preds, masks))
                val_dice += float(calculate_dice(preds, masks))

        n_train, n_val = len(train_loader), len(val_loader)
        avg_train_loss = epoch_loss / n_train
        avg_val_loss = val_loss / n_val
        avg_val_iou = val_iou / n_val
        avg_val_dice = val_dice / n_val

        summary = (
            f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f} | Val IoU: {avg_val_iou:.4f} | "
            f"Val Dice: {avg_val_dice:.4f}"
        )
        print(summary)

        # --- Checkpointing: latest + best only ---
        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": avg_train_loss,
            "val_dice": avg_val_dice,
        }
        torch.save(checkpoint, os.path.join(checkpoint_dir, "denseunet_last.pth"))

        is_best = avg_val_dice > best_dice
        if is_best:
            best_dice = avg_val_dice
            torch.save(checkpoint, os.path.join(checkpoint_dir, "denseunet_best.pth"))
            print(f"  New best Dice: {best_dice:.4f}")

        # --- Discord notifications ---
        if (epoch + 1) % NOTIFY_EVERY == 0 or is_best:
            msg = (
                f"**Epoch {epoch+1}/{EPOCHS}** done\n"
                f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}\n"
                f"Val IoU: {avg_val_iou:.4f} | Val Dice: {avg_val_dice:.4f}"
            )
            if is_best:
                msg += f"\n🏆 New best Dice: {best_dice:.4f}"
            notify_discord(msg)

    notify_discord(f"✅ Training finished. Best Val Dice: {best_dice:.4f}")

except KeyboardInterrupt:
    notify_discord(f"⏹️ Training interrupted at epoch {epoch+1}. Best Val Dice: {best_dice:.4f}")
    raise
except Exception as e:
    notify_discord(f"❌ Training crashed: {type(e).__name__}: {e}")
    raise

Epoch 1/100: 100%|██████████| 300/300 [11:21<00:00,  2.27s/it, loss=0.1439]


Epoch 1/100 | Train Loss: 0.2395 | Val IoU: 0.6251 | Val Dice: 0.7248


Epoch 2/100: 100%|██████████| 300/300 [11:27<00:00,  2.29s/it, loss=0.1086]


Epoch 2/100 | Train Loss: 0.1418 | Val IoU: 0.6466 | Val Dice: 0.7327


Epoch 3/100: 100%|██████████| 300/300 [11:34<00:00,  2.32s/it, loss=0.0643]


...

Epoch 4/100: 100%|██████████| 300/300 [11:08<00:00,  2.23s/it, loss=0.2461]


Epoch 4/100 | Train Loss: 0.1175 | Val IoU: 0.7035 | Val Dice: 0.7907


Epoch 5/100: 100%|██████████| 300/300 [11:08<00:00,  2.23s/it, loss=0.0460]


Epoch 5/100 | Train Loss: 0.1186 | Val IoU: 0.6176 | Val Dice: 0.7155


Epoch 6/100:  95%|█████████▌| 285/300 [10:37<00:33,  2.23s/it, loss=0.2571]

NameError: name 'model' is not defined